# E6 | Model Forecast Prophet
Treinar Prophet para D+1 e D+7 com MLflow

## 📋 Objetivo

Neste notebook, vou treinar um modelo **Prophet** para prever o volume de incidentes com **7 dias de antecedência** (D+1 a D+7). O Prophet é ideal para séries temporais com sazonalidade forte e regressores externos.

### Por que Prophet?
- **Sazonalidade**: Captura padrões semanais (mais incidentes na segunda-feira?) e anuais
- **Regressores**: Incorpora fatores externos (fim de semana, taxa de violação de SLA)
- **Interpretável**: Decompõe a previsão em componentes (trend, sazonal)

### Fluxo
1. **Setup**: Conexão com RDS
2. **Carregamento**: Dados históricos de volume diário
3. **Preparação**: Criação de regressores (fim de semana, taxa de violação)
4. **Treino/Teste**: Split 80/20 nos últimos 30 dias
5. **Avaliação**: MAPE e MAE no conjunto de teste
6. **Forecast**: Previsões para D+1 a D+7
7. **Logging**: Rastreamento no MLflow

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import mlflow

load_dotenv()
print('✅ Imports OK')

Importing plotly failed. Interactive plots will not work.


✅ Imports OK


In [2]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres


## 📥 Etapa 1: Carregamento de Dados

Vou carregar os dados de previsão do RDS. Estes dados já foram processados na camada **gold** do data lake e contêm:
- **data_abertura**: Data do incidente (timestamp)
- **total_chamados**: Volume diário de incidentes (target)
- **is_fim_de_semana**: Indicador de fim de semana (regressador)
- **pct_violacao_sla**: Percentual de violação de SLA (regressador)

### 📊 Dados Carregados

Os dados cobrem **~1 ano de histórico** (2025 inteiro), com 121.263 registros diários. Este volume histórico é essencial para que o Prophet aprenda:
- **Trend**: Tendência geral do volume de incidentes
- **Sazonalidade semanal**: Segundas-feiras têm mais incidentes?
- **Sazonalidade anual**: Períodos de maior carga (Black Friday, férias)?

**Próximo passo**: Vou preparar os dados no formato que o Prophet espera (ds, y, regressadores).

## 🔧 Etapa 2: Preparação dos Dados

O Prophet requer um DataFrame com colunas específicas:
- **ds**: Timestamp (data)
- **y**: Target (volume de incidentes)
- **Regressadores**: Variáveis externas que influenciam o target

Vou criar dois regressadores:
1. **fim_de_semana** (0/1): Permite ao modelo aprender que fins de semana têm padrões diferentes
2. **taxa_violacao** (0-1): A taxa de violação de SLA pode estar correlacionada com volume

## ✂️ Etapa 3: Split Train/Test

Vou usar os **últimos 30 dias** como conjunto de teste. Por quê?
- **Avaliação realista**: Simula prever 30 dias no passado (dados que o modelo não viu)
- **Séries temporais**: Não fazemos split aleatório (quebra a ordem temporal)
- **30 dias**: Período suficiente para avaliar sazonalidade semanal (~4 semanas)

## 🚀 Etapa 4: Treino do Modelo Prophet

Vou configurar o Prophet com:
- **yearly_seasonality=True**: Aprende padrões anuais
- **weekly_seasonality=True**: Aprende padrões semanais (segunda-feira vs domingo)
- **seasonality_mode='additive'**: A sazonalidade é SOMADA ao trend (melhor para volumes absolutos)
- **Regressadores**: Fim de semana e taxa de violação são incorporados ao modelo

O Prophet usa **Stan** (biblioteca de probabilidade) para encontrar os melhores parâmetros. Pode levar alguns segundos.

## 📈 Etapa 5: Avaliação no Conjunto de Teste

Vou usar duas métricas:
- **MAPE** (Mean Absolute Percentage Error): Erro percentual médio. Meta: < 20%
  - Útil para volume de incidentes (entendemos como "20% de erro no volume")
- **MAE** (Mean Absolute Error): Erro absoluto médio em chamados
  - Complementa MAPE: se MAPE=10%, quantos chamados isso representa?

Vou fazer previsão nos 30 dias de teste e comparar com o volume real.

## 🔮 Etapa 6: Forecast D+1 a D+7 (Próxima Semana)

Agora vou fazer **previsões reais** para os próximos 7 dias! O modelo:
1. Usa toda a série histórica (train + test) para treinar novamente
2. Projeta para D+1 a D+7
3. Incorpora:
   - **Trend**: Tendência do volume histórico
   - **Sazonalidade**: Padrão do dia da semana (segunda-feira vs domingo)
   - **Regressadores**: Se segunda é fim de semana (não), qual é a taxa de SLA esperada?

**Output**: Volume previsto para cada dia + intervalo de confiança (95%)

## 📦 Etapa 7: Logging no MLflow e Salvamento de Resultados

Vou:
1. **Rastrear no MLflow**: Registrar parâmetros (sazonalidade), métricas (MAPE, MAE) e artefatos (modelo)
   - Útil para comparar com futuras versões do modelo
   - Auditoria: quem treinou, quando, com quais parâmetros
2. **Salvar resultados locais**:
   - **forecast_d1_d7.csv**: Previsões D+1 a D+7 para usar em dashboards
   - **prophet_test_metrics.csv**: Avaliação no teste (erros por dia)
   - **prophet_summary.csv**: Resumo de métricas

In [3]:
# Carregar dados para XGBoost
from sqlalchemy import create_engine

# Reutilizar credenciais da célula anterior (RDS_HOST, RDS_USER, RDS_PASSWORD, RDS_DATABASE)
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

In [ ]:
# Ler e agregar dados por dia (CORRIGIDO)
query = '''
    SELECT DATE(data_abertura) AS data_abertura,
           COUNT(*) AS total_chamados,
           MAX(CAST(is_fim_de_semana AS INT)) AS is_fim_de_semana,
           AVG(CAST(pct_violacao_sla AS FLOAT)) AS pct_violacao_sla
    FROM gold_ml.ml_forecast_dataset
    GROUP BY DATE(data_abertura)
    ORDER BY data_abertura
'''
df = pd.read_sql(query, engine)
df['data_abertura'] = pd.to_datetime(df['data_abertura'])
print(f'✅ Loaded {len(df)} DIAS from {df.data_abertura.min().date()} to {df.data_abertura.max().date()}')
print(f'   Volume médio: {df.total_chamados.mean():.0f} chamados/dia')
print(f'   Min/Max: {df.total_chamados.min():.0f} / {df.total_chamados.max():.0f}')

In [6]:
# Preparar Prophet dataset
df_p = df[['data_abertura', 'total_chamados']].rename(columns={'data_abertura': 'ds', 'total_chamados': 'y'})
df_p['ds'] = pd.to_datetime(df_p['ds'])
df_p['fim_de_semana'] = df['is_fim_de_semana'].values
df_p['taxa_violacao'] = df['pct_violacao_sla'].fillna(df['pct_violacao_sla'].mean()).values
df_p = df_p.sort_values('ds').reset_index(drop=True)
print(f'Ready: {df_p.shape}')

Ready: (121263, 4)


In [7]:
# Split
train = df_p.iloc[:-30]
test = df_p.iloc[-30:]
print(f'Train: {len(train)}, Test: {len(test)}')

Train: 121233, Test: 30


In [8]:
# Treinar
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, seasonality_mode='additive')
model.add_regressor('fim_de_semana')
model.add_regressor('taxa_violacao')
print('Training...')
model.fit(train)
print('✅ Done')

Training...


09:02:03 - cmdstanpy - INFO - Chain [1] start processing
09:02:13 - cmdstanpy - INFO - Chain [1] done processing


✅ Done


In [10]:
# Avaliar
forecast = model.predict(test[['ds', 'fim_de_semana', 'taxa_violacao']])
eval = test[['y']].copy()
eval['yhat'] = forecast['yhat'].values

mape = mean_absolute_percentage_error(eval['y'], eval['yhat'])
mae = (eval['y'] - eval['yhat']).abs().mean()
print(f'MAPE: {mape:.2%}, MAE: {mae:.0f}')

MAPE: 0.42%, MAE: 0


In [11]:
# Forecast D+1 a D+7
future_dates = pd.date_range(start=pd.Timestamp(datetime.now()).normalize() + timedelta(days=1), periods=7)
future = pd.DataFrame({
    'ds': future_dates,
    'fim_de_semana': [1 if d.dayofweek in [5,6] else 0 for d in future_dates],
    'taxa_violacao': df_p['taxa_violacao'].mean()
})

forecast_future = model.predict(future)
for i, row in forecast_future.iterrows():
    yhat = int(row['yhat'])
    date = row['ds'].strftime('%d/%m')
    print(f'D+{i+1} ({date}): {yhat} chamados')

D+1 (21/05): 1 chamados
D+2 (22/05): 1 chamados
D+3 (23/05): 1 chamados
D+4 (24/05): 1 chamados
D+5 (25/05): 1 chamados
D+6 (26/05): 1 chamados
D+7 (27/05): 1 chamados


In [12]:
# MLflow
import joblib
import tempfile

mlflow.set_experiment('prophet_forecast')
with mlflow.start_run():
    mlflow.log_params({'seasonality': 'additive', 'train_days': len(train)})
    mlflow.log_metrics({'mape': mape, 'mae': mae})
    # Salvar modelo com tempfile (compatível com Windows)
    temp_path = os.path.join(tempfile.gettempdir(), 'prophet_model.pkl')
    joblib.dump(model, temp_path)
    mlflow.log_artifact(temp_path, 'model')
    print('✅ Logged')

✅ Logged
🏃 View run funny-penguin-84 at: https://mlflow.looplyai.com.br/#/experiments/10/runs/61f9adc16e2440d0be090df4024b48ef
🧪 View experiment at: https://mlflow.looplyai.com.br/#/experiments/10


In [14]:
# Salvar resultados em data/ml/ (com caminho hardcoded)
import os
from pathlib import Path

# Usar caminho ABSOLUTO (hardcoded)
base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast')
base_path.mkdir(parents=True, exist_ok=True)

print(f'Salvando em: {base_path}')
print(f'Pasta existe: {base_path.exists()}')
print(f'Dados disponíveis:')
print(f'  - eval shape: {eval.shape}')
print(f'  - forecast_future shape: {forecast_future.shape}')

# Salvar forecast D+1 a D+7
try:
    forecast_d = forecast_future[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_d.columns = ['data', 'forecast', 'lower_ci', 'upper_ci']
    forecast_d['data'] = forecast_d['data'].dt.strftime('%Y-%m-%d')
    forecast_d['forecast'] = forecast_d['forecast'].round(0).astype(int)
    filepath = base_path / 'forecast_d1_d7.csv'
    forecast_d.to_csv(str(filepath), index=False)
    print(f'✅ Saved D+1 to D+7: {filepath}')
except Exception as e:
    print(f'❌ Erro ao salvar forecast: {e}')

# Salvar métricas de avaliação no teste
try:
    eval_results = eval.copy()
    eval_results['erro_pct'] = ((eval_results['y'] - eval_results['yhat']).abs() / (eval_results['y'] + 1) * 100).fillna(0)
    filepath = base_path / 'prophet_test_metrics.csv'
    eval_results.to_csv(str(filepath), index=False)
    print(f'✅ Saved test metrics: {filepath}')
except Exception as e:
    print(f'❌ Erro ao salvar métricas: {e}')

# Resumo
try:
    summary = pd.DataFrame({
        'métrica': ['MAPE', 'MAE', 'Dados de treino', 'Dados de teste'],
        'valor': [f'{mape:.2%}', f'{mae:.0f}', len(train), len(test)]
    })
    filepath = base_path / 'prophet_summary.csv'
    summary.to_csv(str(filepath), index=False)
    print(f'✅ Saved summary: {filepath}')
except Exception as e:
    print(f'❌ Erro ao salvar resumo: {e}')

# Verificar se os arquivos foram criados
print('\n📋 Arquivos criados:')
for file in base_path.glob('*.csv'):
    print(f'  ✅ {file.name}')

Salvando em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast
Pasta existe: True
Dados disponíveis:
  - eval shape: (30, 2)
  - forecast_future shape: (7, 31)
✅ Saved D+1 to D+7: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast\forecast_d1_d7.csv
✅ Saved test metrics: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast\prophet_test_metrics.csv
✅ Saved summary: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast\prophet_summary.csv

📋 Arquivos criados:
  ✅ forecast_d1_d7.csv
  ✅ prophet_summary.csv
  ✅ prophet_test_metrics.csv
